In [13]:
import os
import json
import openai
import re
import random
import importlib
import matplotlib.pyplot as plt
from openai import OpenAI

In [14]:
# Load the API key into the client for OpenAI API access.
def load_api_key(file_path='api_key.txt'):
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith('api_key='):
                return line.strip().split('=', 1)[1]
    return None

api_key = load_api_key()

if api_key is None:
    print("Error: API key not found.")
    exit()

client = OpenAI(api_key=api_key,)

chat_completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "Say this is a test",
        }
    ]
)

In [15]:
# Load prompts from the file
def load_prompts_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()  # Read the entire file content

    # Split the content into system and user sections using markers
    system_prompt = content.split('--USER--')[0].replace('--SYSTEM--', '').strip()
    user_prompt = content.split('--USER--')[1].strip()
    
    return system_prompt, user_prompt

In [16]:
def load_env_objects(env_id: str, filename: str = "room_objects.json") -> dict:
    with open(filename, 'r') as f:
        all_envs = json.load(f)
    
    if env_id not in all_envs:
        raise ValueError(f"{env_id} not found in {filename}")
    
    return all_envs[env_id]

In [17]:
# Validate the activity's room and generate a detailed routine breakdown
# This function determines the most appropriate room for the activity
# and then uses an LLM prompt to generate fine-grained routine steps
def validate_and_generate_routine(start_time, end_time, activity, description, room_list, object_dictionary, prompt_path, client):
    # Determine which room the activity should take place in
    selected_room = determine_activity_room(activity, room_list, client)
    # Generate a detailed routine breakdown for the activity
    routine_content = schedule_breakdown(start_time, end_time, activity, description, selected_room, object_dictionary, prompt_path, client)
    return selected_room, routine_content

In [18]:
def determine_activity_room(activity, room_list, client):
    room_list_str = ', '.join(room_list)
    valid_rooms = set(room_list)
    max_attempts = 10
    attempt = 0

    system_prompt = (
        "You will be provided with a list of rooms and an activity name. "
        "Your task is to determine which room is most likely to be the location for the given activity."
    )
    
    user_prompt = (
        "Your task is to determine the most appropriate room for a given activity from the provided list. "
        "Below are the activity description and the room list:\n\n"
        f"Activity: {activity}\n"
        f"Room list: {room_list_str}\n\n"
        "Return the name of the room that is most suitable for this activity from the list above. "
        "You must select and return exactly one room name. Do not include any explanations or additional information, just the room name."
    )

    while attempt < max_attempts:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )

        selected_room = response.choices[0].message.content.strip().lower()
        selected_room = selected_room.replace(" ", "")  # Normalize by removing spaces

        if selected_room in valid_rooms:
            return selected_room  # Return the valid room name

        print(f"Attempt {attempt + 1}: Invalid room detected ({selected_room}), retrying...")
        attempt += 1

    print("Max attempts reached. Returning 'bedroom' as default.")
    return "bedroom"  # Default value if all attempts fail


In [19]:
def schedule_breakdown(start_time, end_time, activity, description, selected_room, object_dictionary, prompt_path, client):
    system_prompt, user_prompt = load_prompts_from_file(prompt_path)
    
    # Retrieve objects belonging to the selected room
    room_objects = object_dictionary.get(selected_room, [])
    objects_str = ', '.join(room_objects)

    # Construct the prompt    
    prompt = (
        "Now you are provided with the following details:\n\n"
        f"Activity Name: {activity}\n"
        f"Start Time: {start_time}\n"
        f"End Time: {end_time}\n"
        f"Personality Description: {description}\n"
        f"Location: {selected_room}\n\n"
        "Based on the 'Activity Name,' "
        "break this activity into detailed action steps corresponding to smaller time intervals (24-hour format) within the 'Start Time' and 'End Time'. "
        "For each smaller time interval, use the following format:\n\n"
        f"First line: 'Start time - End time, {selected_room}'\n\n"
        "Then, in the following lines, describe the activities during that smaller time interval. "
        "Each action step should also have its own time slot. "
        "The 'Personality Description' is provided for reference. "
        f"Each activity step can only use objects from this list: {objects_str}.\n\n"
    )


    user_prompt = prompt + user_prompt
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content.strip()

In [20]:
# Load a single weekly schedule file and return (character_name, schedule_text)
# Expected filename format: {character_name}_weekly_schedule.txt
def load_schedule_from_file(schedule_file_path: str) -> tuple[str, str]:

    filename = os.path.basename(schedule_file_path)
    character_name = filename.split("_weekly_schedule")[0]

    with open(schedule_file_path, "r", encoding="utf-8") as f:
        schedule_text = f.read().strip()

    return character_name, schedule_text

# Load a single personality file and return (character_name, personality_text)
# Expected filename format: {character_name}_personality.txt
def load_personality_from_file(personality_file_path: str) -> tuple[str, str]:

    filename = os.path.basename(personality_file_path)
    character_name = filename.split("_personality")[0]

    with open(personality_file_path, "r", encoding="utf-8") as f:
        personality_text = f.read().strip()

    return character_name, personality_text


In [21]:
def extract_time_slots_by_day_at_home(schedule_text):
    # Regex patterns to identify days and activity time slots
    days_pattern = re.compile(r'(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday):')
    time_slot_pattern = re.compile(r'([a-zA-Z_]+) \((\d{2}:\d{2}) - (\d{2}:\d{2}(?: \d+day)?(?: \d{2}:\d{2})?)\) \((at home|outside)\)')
    
    # Regex patterns to identify days and activity time slots
    time_slots_by_day = {}

     # Split the full schedule into day-specific sections
    schedule_parts = re.split(days_pattern, schedule_text)

    # Iterate through each day and its corresponding schedule
    for i in range(1, len(schedule_parts), 2):
        day = schedule_parts[i].strip()     # Day name (e.g., Monday)
        day_schedule = schedule_parts[i+1]  # Schedule content for that day
        
        # Extract all activity time slots for the day
        matches = time_slot_pattern.findall(day_schedule)

        # Keep only activities that occur "at home"
        time_slots_at_home = [(match[0], match[1], match[2], match[3]) for match in matches if match[3] == "at home"]
        
        # Store the day's at-home activities if any exist
        if time_slots_at_home:
            time_slots_by_day[day] = time_slots_at_home

    return time_slots_by_day

In [22]:
# Function to process each character's "at home" time slots and generate routines
def process_home_activities_for_character(character_name, schedule_text, description, room_list, object_dictionary, env_id, output_dir, routine_prompt_path, client):
    # Extract all at-home time slots from the schedule
    time_slots_by_day = extract_time_slots_by_day_at_home(schedule_text)

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Filename for each character's routine
    routine_filename = f"{character_name}_routine_{env_id}.txt"
    routine_filepath = os.path.join(output_dir, routine_filename)
    
    # Write routines to file
    with open(routine_filepath, 'a', encoding='utf-8') as file_out:
        for day, time_slots in time_slots_by_day.items():
            file_out.write(f"{day}:\n")
            
            for activity, start_time, end_time, location in time_slots:
                # Generate a detailed breakdown for each time slot using schedule_breakdown_one
                room, routine_content = validate_and_generate_routine(
                    start_time,
                    end_time,
                    activity,
                    description,
                    room_list,
                    object_dictionary,
                    routine_prompt_path,
                    client
                )                
                file_out.write(f"{start_time} - {end_time}, {room}: {activity}\n\n")
                file_out.write(f"{routine_content}\n\n")
    
    print(f"[AgentSense] Routine saved to {routine_filepath.replace(os.sep, '/')}")

In [23]:
# Load one schedule + one personality file, then generate routines for that character
def process_home_activities_from_files(schedule_file_path: str, personality_file_path: str, env_id: str, output_dir: str, routine_prompt_path, client):

    # Load schedule + personality
    schedule_character, schedule_text = load_schedule_from_file(schedule_file_path)
    personality_character, personality_text = load_personality_from_file(personality_file_path)

    # Sanity check: names should match
    if schedule_character != personality_character:
        print(
            f"[Warning] Character name mismatch:\n"
            f"  Schedule file -> {schedule_character}\n"
            f"  Personality file -> {personality_character}\n"
            f"Proceeding with schedule character = {schedule_character}."
        )

    character_name = schedule_character
    description = personality_text

    print(
        f"[AgentSense] Generating routines for character '{character_name}' "
        f"in environment '{env_id}'..."
    )

    # Room list for each character's home (do not modify this list; household activity generation assumes these four rooms)
    room_list = ["bathroom", "kitchen", "bedroom", "livingroom"]

    # Load environment objects
    object_dictionary = load_env_objects(env_id)

    process_home_activities_for_character(
        character_name=character_name,
        schedule_text=schedule_text,
        description=description,
        room_list=room_list,
        object_dictionary=object_dictionary,
        env_id=env_id,
        output_dir=output_dir,
        routine_prompt_path=routine_prompt_path,
        client=client
    )


In [24]:
# =========================
# Configuration (User-editable)
# =========================

# Select ONE VirtualHome environment to run
# Recommended environments:
# ["env_0", "env_8", "env_9", "env_18", "env_28", "env_29", "env_32", "env_33", "env_35", "env_43"]
#
# These environments are recommended because:
# 1. They contain exactly one bedroom, one bathroom, one kitchen, and one living room
# 2. Their layouts are compatible with the generated schedules and room constraints
# 3. They run smoothly in the VirtualHome simulator and rarely fail during execution
ENV_ID = "env_0"

# Select ONE personality file and ONE corresponding schedule file
# Make sure both files belong to the same synthetic character
PERSONALITY_FILE = "data/personality_data/Sarah_personality.txt"
SCHEDULE_FILE = "data/schedule_data/Sarah_weekly_schedule.txt"

# Path to the routine generation prompt
ROUTINE_PROMPT_PATH = "prompts/routine_generation_prompt.txt"

# Directory to save generated routines
ROUTINE_OUTPUT_DIR = "data/step_3_routine_data"

# =========================
# Run
# =========================

process_home_activities_from_files(
    schedule_file_path=SCHEDULE_FILE,
    personality_file_path=PERSONALITY_FILE,
    env_id=ENV_ID,
    output_dir=ROUTINE_OUTPUT_DIR,
    routine_prompt_path=ROUTINE_PROMPT_PATH,
    client=client
);

[AgentSense] Generating routines for character 'Sarah' in environment 'env_0'...
[AgentSense] Routine saved to data/routine_data/Sarah_routine_env_0.txt
